---
title: "Causal Forests: From Estimation to Analysis"
subtitle: "HPM 883 — Session 3.0 Code-Along"
author: "Sean Sylvia"
date: "March 25, 2026"
format:
  html:
    toc: true
    toc-depth: 3
    code-fold: false
    code-tools: true
    theme: cosmo
    self-contained: true
execute:
  echo: true
  warning: false
  message: false
  eval: false
---

## Overview

This notebook walks through the **complete causal forest analysis pipeline** using the `grf` package:

1. **Estimate** CATEs with `causal_forest()`
2. **Test** whether heterogeneity is real with `test_calibration()`
3. **Summarize** heterogeneity with `best_linear_projection()`
4. **Visualize** who benefits most (sorted group analysis)
5. **Evaluate** targeting quality with RATE/AUTOC
6. **Preview** policy learning with `policytree`

Based on the framework from Athey, Tibshirani & Wager (2019) and the [Stanford GSB ML-CI Tutorial](https://bookdown.org/stanfordgsbsilab/ml-ci-tutorial/hte-i-binary-treatment.html).

## Setup

```{r}
#| label: setup

# Core packages
library(grf)
library(policytree)

# Visualization and inference
library(ggplot2)
library(sandwich)
library(lmtest)

theme_set(theme_minimal(base_size = 14))
```

## 1. Simulated Data

We simulate an RCT where treatment effects vary by patient characteristics.

**Setting:** A community health worker (CHW) program reduces ER visits. But the effect depends on comorbidity burden and age.

```{r}
#| label: simulate-data

set.seed(883)
n <- 2000
p <- 6

# Patient characteristics
age <- runif(n, 25, 75)
comorbidities <- rpois(n, lambda = 1.5)
female <- rbinom(n, 1, 0.52)
income_quartile <- sample(1:4, n, replace = TRUE)
rural <- rbinom(n, 1, 0.35)
prior_er_visits <- rpois(n, lambda = 3)

X <- data.frame(
  age = age,
  comorbidities = comorbidities,
  female = female,
  income_quartile = income_quartile,
  rural = rural,
  prior_er_visits = prior_er_visits
)

# Treatment assignment (RCT: 50/50)
W <- rbinom(n, 1, 0.5)

# True CATE function: effect depends on comorbidities and age
# Higher comorbidities → larger benefit
# Older patients → larger benefit
# Base effect is small
tau.true <- -0.5 + -0.8 * (comorbidities >= 2) + -0.6 * (age > 55)

# Outcome: ER visits in 12 months
Y <- 4 + 0.3 * comorbidities + 0.02 * age - 0.5 * female +
  tau.true * W + rnorm(n, 0, 2)

cat("True ATE:", round(mean(tau.true), 3), "\n")
cat("True CATE range:", round(range(tau.true), 3), "\n")
```

## 2. Estimate CATEs with Causal Forest

```{r}
#| label: fit-causal-forest

# Fit a causal forest
# W.hat = 0.5 because this is an RCT with known assignment probability
cf <- causal_forest(
  X = as.matrix(X),
  Y = Y,
  W = W,
  W.hat = 0.5,        # Known propensity (RCT)
  num.trees = 2000,
  seed = 883
)

# Out-of-bag CATE predictions
tau.hat <- predict(cf)$predictions

# Summary
cat("Estimated ATE (forest-weighted):\n")
average_treatment_effect(cf)

cat("\nCATE distribution:\n")
summary(tau.hat)
```

```{r}
#| label: cate-distribution

# Visualize the CATE distribution
ggplot(data.frame(tau = tau.hat), aes(x = tau)) +
  geom_histogram(bins = 40, fill = "#4B9CD3", color = "white", alpha = 0.8) +
  geom_vline(xintercept = mean(tau.hat), color = "#E07C3E",
             linewidth = 1.2, linetype = "dashed") +
  labs(
    title = "Distribution of Estimated CATEs",
    subtitle = "Orange dashed line = ATE (forest-weighted average)",
    x = expression(hat(tau)(x)),
    y = "Count"
  ) +
  annotate("text", x = mean(tau.hat) + 0.15, y = Inf, vjust = 2,
           label = paste("ATE =", round(mean(tau.hat), 2)),
           color = "#E07C3E", fontface = "bold")
```

::: {.callout-note}
## What are we looking at?
The histogram shows the predicted treatment effect for each individual. If the distribution is tight around the ATE, there's little heterogeneity. A wide spread suggests meaningful variation in who benefits.
:::

## 3. Test: Is the Heterogeneity Real?

Before interpreting CATEs, we need to check that the forest is detecting **real** heterogeneity, not just noise.

### 3a. Calibration Test

```{r}
#| label: test-calibration

# Omnibus test for heterogeneous treatment effects
cal <- test_calibration(cf)
print(cal)
```

::: {.callout-tip}
## How to read this
- **mean.forest.prediction** ≈ 1: the forest's ATE estimate is well-calibrated
- **differential.forest.prediction** > 0 with p < 0.05: **real heterogeneity detected**
- If the differential coefficient is not significant, the "heterogeneity" may be noise
:::

### 3b. Variable Importance

Which covariates does the forest use most for splitting?

```{r}
#| label: variable-importance

var.imp <- variable_importance(cf)
var.imp.df <- data.frame(
  variable = colnames(X),
  importance = as.numeric(var.imp)
)
var.imp.df <- var.imp.df[order(-var.imp.df$importance), ]

ggplot(var.imp.df, aes(x = reorder(variable, importance), y = importance)) +
  geom_col(fill = "#4B9CD3") +
  coord_flip() +
  labs(
    title = "Variable Importance for Treatment Effect Heterogeneity",
    subtitle = "How often each variable is used in forest splits",
    x = NULL, y = "Importance (weighted split frequency)"
  )
```

::: {.callout-warning}
## Caution
Variable importance tells you which variables the forest **splits on**, not the direction or magnitude of the effect. Use BLP (next step) for interpretable summaries.
:::

## 4. Summarize: Who Benefits Most?

### 4a. Best Linear Projection (BLP)

BLP projects the CATEs onto covariates of interest, giving interpretable coefficients with standard errors.

```{r}
#| label: blp

# Project CATEs onto top variables (informed by variable importance)
blp <- best_linear_projection(
  cf,
  A = as.matrix(X[, c("comorbidities", "age", "female")])
)
print(blp)
```

::: {.callout-note}
## Interpreting BLP
Each coefficient tells you: "A one-unit increase in this covariate is associated with this much change in the treatment effect." For example, if the comorbidities coefficient is -0.7, each additional comorbidity increases the treatment *benefit* by 0.7 fewer ER visits.

This is NOT a causal claim about the covariate — it's a summary of heterogeneity patterns.
:::

### 4b. Sorted Group Analysis (GATES-style)

Rank individuals by predicted CATE, split into quintiles, and estimate the ATE within each group.

```{r}
#| label: sorted-groups

# Create quintile rankings using K-fold approach (avoids overfitting)
num.rankings <- 5
num.folds <- 5
folds <- sample(rep(1:num.folds, length.out = n))

ranking <- rep(NA, n)
for (fold in seq(num.folds)) {
  in.fold <- (folds == fold)
  tau.fold <- tau.hat[in.fold]
  breaks <- quantile(tau.fold, probs = seq(0, 1, by = 1 / num.rankings))
  ranking[in.fold] <- as.numeric(cut(tau.fold, breaks,
                                      include.lowest = TRUE,
                                      labels = seq(num.rankings)))
}

# Compute doubly robust ATE within each quintile
# Using AIPW scores from the forest
e.hat <- cf$W.hat   # propensity (known = 0.5 in RCT)
m.hat <- cf$Y.hat   # outcome model
mu.hat.0 <- m.hat - e.hat * tau.hat
mu.hat.1 <- m.hat + (1 - e.hat) * tau.hat

aipw.scores <- tau.hat +
  W / e.hat * (Y - mu.hat.1) -
  (1 - W) / (1 - e.hat) * (Y - mu.hat.0)

# OLS with quintile dummies (no intercept)
ols <- lm(aipw.scores ~ 0 + factor(ranking))
gates <- coeftest(ols, vcov = vcovHC(ols, "HC2"))
print(gates)
```

```{r}
#| label: gates-plot

# Visualize GATES
gates.df <- data.frame(
  group = 1:num.rankings,
  ate = coef(ols),
  se = sqrt(diag(vcovHC(ols, "HC2")))
)
gates.df$lower <- gates.df$ate - 1.96 * gates.df$se
gates.df$upper <- gates.df$ate + 1.96 * gates.df$se

ggplot(gates.df, aes(x = factor(group), y = ate)) +
  geom_point(size = 3, color = "#13294B") +
  geom_errorbar(aes(ymin = lower, ymax = upper), width = 0.2, color = "#13294B") +
  geom_hline(yintercept = mean(tau.hat), linetype = "dashed", color = "#E07C3E") +
  labs(
    title = "Group Average Treatment Effects (GATES)",
    subtitle = "ATE within each quintile of predicted CATE",
    x = "CATE Quintile (1 = least benefit, 5 = most benefit)",
    y = "Estimated ATE (AIPW)"
  ) +
  annotate("text", x = 0.7, y = mean(tau.hat), vjust = -1,
           label = "Overall ATE", color = "#E07C3E", fontface = "italic")
```

::: {.callout-note}
## What are GATES telling us?
If the bars are roughly equal across groups, there's little heterogeneity. If Group 5 (highest predicted CATE) has a much larger effect than Group 1, the forest is successfully identifying who benefits most.
:::

### 4c. CLAN: Who is in each group?

Compare covariate means between the most-affected and least-affected groups.

```{r}
#| label: clan

# Compare covariate means: Group 5 (most benefit) vs Group 1 (least benefit)
top.group <- ranking == num.rankings
bottom.group <- ranking == 1

clan.results <- data.frame(
  variable = colnames(X),
  mean.bottom = sapply(X[bottom.group, ], mean),
  mean.top = sapply(X[top.group, ], mean)
)
clan.results$difference <- clan.results$mean.top - clan.results$mean.bottom

print(clan.results)
```

```{r}
#| label: clan-plot

ggplot(clan.results, aes(x = reorder(variable, difference), y = difference)) +
  geom_col(fill = ifelse(clan.results$difference > 0, "#4B9CD3", "#E07C3E")) +
  coord_flip() +
  labs(
    title = "CLAN: How do the most-affected differ from least-affected?",
    subtitle = "Difference in covariate means: Top quintile − Bottom quintile",
    x = NULL, y = "Difference in means"
  )
```

::: {.callout-note}
## Interpreting CLAN
This tells you what **characterizes** the people who benefit most. For example, if comorbidities are higher in the top group, it suggests the intervention is most beneficial for sicker patients. This is descriptive, not causal.
:::

## 5. Evaluate: How Good is Our Targeting?

### RATE and TOC Curves

The **Rank-Average Treatment Effect (RATE)** evaluates how well our CATE predictions can be used for targeting.

```{r}
#| label: rate

# Split-sample approach for honest evaluation
train.idx <- sample(1:n, n / 2)
test.idx <- setdiff(1:n, train.idx)

# Train forest on training set
cf.train <- causal_forest(
  as.matrix(X[train.idx, ]), Y[train.idx], W[train.idx],
  W.hat = 0.5, num.trees = 2000, seed = 883
)

# Predict CATEs on test set
tau.hat.test <- predict(cf.train, as.matrix(X[test.idx, ]))$predictions

# Evaluation forest on test set
cf.eval <- causal_forest(
  as.matrix(X[test.idx, ]), Y[test.idx], W[test.idx],
  W.hat = 0.5, num.trees = 2000, seed = 884
)

# RATE: AUTOC (recommended default)
rate.result <- rank_average_treatment_effect(cf.eval, tau.hat.test)
print(rate.result)

# One-sided p-value
cat("\nOne-sided p-value:",
    pnorm(rate.result$estimate / rate.result$std.err, lower.tail = FALSE), "\n")
```

```{r}
#| label: toc-curve

# TOC curve (Targeting Operator Characteristic)
plot(rate.result, las = 1,
     main = "TOC Curve: Targeting by Predicted CATE",
     xlab = "Fraction treated (highest CATE first)",
     ylab = "Treatment effect advantage")
```

::: {.callout-tip}
## How to read the TOC curve
The TOC curve shows what happens if you treat individuals **in order of predicted benefit** (highest CATE first). The y-axis is the treatment effect for the treated fraction minus the ATE. If the curve is above zero on the left, your targeting rule successfully identifies high-benefit individuals.

- **Area under the curve (AUTOC)** > 0 with p < 0.05: targeting works
- If the curve is flat at zero: no useful heterogeneity for targeting
:::

## 6. Preview: Policy Learning

Given CATEs, can we learn a simple, interpretable treatment rule?

```{r}
#| label: policy-tree

library(policytree)

# Extract doubly robust scores (these encode the causal information)
dr.scores <- double_robust_scores(cf)

# Learn a shallow policy tree (depth = 2 for interpretability)
ptree <- policy_tree(as.matrix(X), dr.scores, depth = 2)

# Visualize the policy
plot(ptree, leaf.labels = c("Don't Treat", "Treat"))
```

::: {.callout-note}
## What is this?
The policy tree learns a simple decision rule: "Treat if comorbidities > X AND age > Y." It maximizes expected welfare under the learned rule. We'll formalize this in **Unit 4: Policy Learning**.
:::

## 7. Putting It All Together

Here's the recommended analysis workflow:

| Step | Function | Question Answered |
|------|----------|-------------------|
| 1. Estimate | `causal_forest()` + `predict()` | What is each person's treatment effect? |
| 2. Test | `test_calibration()` | Is the heterogeneity real or noise? |
| 3. Screen | `variable_importance()` | Which covariates drive heterogeneity? |
| 4. Summarize | `best_linear_projection()` | How do covariates relate to CATEs? |
| 5. Visualize | Sorted group analysis (GATES) | How do ATEs vary across CATE quintiles? |
| 6. Characterize | CLAN analysis | Who is in the high-benefit group? |
| 7. Evaluate | `rank_average_treatment_effect()` | Can we target effectively? |
| 8. Decide | `policy_tree()` | What's the optimal treatment rule? |

## References

- Athey, S., Tibshirani, J., & Wager, S. (2019). Generalized Random Forests. *Annals of Statistics*, 47(2), 1148-1178.
- Wager, S. (2024). *Causal Inference: A Statistical Learning Approach*. Chapter 6.
- Chernozhukov, V. et al. (2025). *Applied Causal Inference Powered by ML and AI*. Chapter 8.
- [Stanford GSB ML-CI Tutorial](https://bookdown.org/stanfordgsbsilab/ml-ci-tutorial/hte-i-binary-treatment.html)
- [GRF Documentation](https://grf-labs.github.io/grf/)
- [GRF Diagnostics Vignette](https://grf-labs.github.io/grf/articles/diagnostics.html)
- [GRF RATE Vignette](https://grf-labs.github.io/grf/articles/rate.html)